# FlashAlpha Historical — Getting Started

Point-in-time replay of every live FlashAlpha analytics endpoint. Ask what dealer positioning, gamma flip, VRP, narrative, max pain, or the full stock summary looked like at any **minute back to 2018-04-16**.

```bash
pip install flashalpha-historical
```

Set your API key in the environment first (same key as live FlashAlpha — Alpha plan or higher required):

```bash
export FLASHALPHA_API_KEY=fa_your_key_here
```

In [ ]:
import os
import pandas as pd

from flashalpha_historical import (
    Backtester,
    FlashAlphaHistorical,
    iter_days,
    iter_minutes,
    replay,
)

hx = FlashAlphaHistorical(os.environ["FLASHALPHA_API_KEY"])

## 1. Coverage check

What's loaded, healthy day count, and any gaps?

In [ ]:
spy_cov = hx.tickers(symbol="SPY")
print(f"SPY: {spy_cov['coverage']['first']} → {spy_cov['coverage']['last']}")
print(f"healthy days: {spy_cov['coverage']['healthy_days']:,}")
print(f"gaps:        {spy_cov['gaps']}")

## 2. Single-minute snapshot

What did dealer positioning look like at the COVID-crash close on March 16, 2020?

In [ ]:
snap = hx.exposure_summary("SPY", at="2020-03-16T15:30:00")
print(f"SPY @ {snap['as_of']}")
print(f"  spot:        {snap['underlying_price']}")
print(f"  regime:      {snap['regime']}")
print(f"  net GEX:     ${snap['exposures']['net_gex']:,}")
print(f"  gamma flip:  {snap['gamma_flip']}")

## 3. Daily backtest — VRP harvest signal

Walk SPY day by day, pull the full stock summary at session close, and flag days where VRP > 5 vol points and dealers are long gamma.

In [ ]:
def harvest(at, snap):
    vrp = snap["volatility"]["vrp"]
    regime = snap["exposure"]["regime"]
    return {
        "fire": vrp is not None and vrp > 5 and regime == "positive_gamma",
        "vrp": vrp,
        "regime": regime,
        "atm_iv": snap["volatility"]["atm_iv"],
    }

bt = Backtester(hx, method="stock_summary", symbol="SPY")
results = bt.run(iter_days("2024-01-02", "2024-03-29"), harvest)
df = pd.DataFrame(bt.to_records(results))
df.head()

In [ ]:
fires = df[df["fire"]]
print(f"{len(fires)} signal fires across {len(df)} days")
fires[["at", "underlying_price", "vrp", "regime", "atm_iv"]].head(10)

## 4. Intraday gamma-flip tracking

Replay one day at 15-minute resolution and watch SPY's gamma flip move relative to spot.

In [ ]:
rows = []
for at, snap in replay(
    hx,
    "exposure_summary",
    "SPY",
    iter_minutes("2024-08-05", "2024-08-05", step_minutes=15),
):
    rows.append({
        "at": at,
        "spot": snap["underlying_price"],
        "flip": snap["gamma_flip"],
        "net_gex": snap["exposures"]["net_gex"],
        "regime": snap["regime"],
    })
intraday = pd.DataFrame(rows)
intraday["gap"] = intraday["spot"] - intraday["flip"]
intraday

## 5. Quota note

Every call counts against your daily plan quota (shared with the live FlashAlpha API). 1-minute replay = 390 calls per analytic per day. Coarsen with `step_minutes=15` or `30` during development.